In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_squared_error, r2_score, roc_auc_score, roc_curve,
    ConfusionMatrixDisplay
)
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
import joblib
import os

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# GLOBAL STYLE
# ─────────────────────────────────────────────────────────────────────────────
PALETTE = {
    "low":    "#2ecc71",
    "medium": "#f39c12",
    "high":   "#e74c3c",
    "bg":     "#0d0d0d",
    "card":   "#1a1a2e",
    "accent": "#e94560",
    "text":   "#eaeaea",
}
RISK_CMAP = LinearSegmentedColormap.from_list(
    "risk", ["#2ecc71", "#f39c12", "#e74c3c"], N=256
)

plt.rcParams.update({
    "figure.facecolor": PALETTE["bg"],
    "axes.facecolor":   "#111122",
    "axes.edgecolor":   "#333355",
    "axes.labelcolor":  PALETTE["text"],
    "xtick.color":      PALETTE["text"],
    "ytick.color":      PALETTE["text"],
    "text.color":       PALETTE["text"],
    "grid.color":       "#222244",
    "grid.alpha":       0.4,
    "font.family":      "monospace",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
})

OUTPUT_DIR = "/mnt/user-data/outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save(fig, name):
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  ✔  Saved → {name}")

def section(title):
    bar = "═" * 70
    print(f"\n{bar}")
    print(f"  {title}")
    print(f"{bar}")

In [3]:
# ═════════════════════════════════════════════════════════════════════════════
# 1.  DATA LOADING & SUMMARY
# ═════════════════════════════════════════════════════════════════════════════
section("1. DATA LOADING & SUMMARY")

DATA_PATH = "womensafety_updated.csv"
df = pd.read_csv(DATA_PATH)

print("\n── head(5) ──────────────────────────────────────────────────────────")
print(df.head(5).to_string())

print("\n── shape ────────────────────────────────────────────────────────────")
print(f"  Rows: {df.shape[0]:,}   Columns: {df.shape[1]}")

print("\n── columns ──────────────────────────────────────────────────────────")
for i, c in enumerate(df.columns, 1):
    print(f"  {i:>2}. {c}")

print("\n── info() ───────────────────────────────────────────────────────────")
df.info()

print("\n── describe() ───────────────────────────────────────────────────────")
print(df.describe().to_string())

print("\n── missing values ───────────────────────────────────────────────────")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "  No missing values ✔")

print("\n── target distribution ──────────────────────────────────────────────")
print(df["risk_level"].value_counts())
print(df["risk_level"].value_counts(normalize=True).mul(100).round(2))

print("\n── unique area names ─────────────────────────────────────────────────")
print(f"  {df['area_name'].nunique()} unique areas")
print(df["area_name"].value_counts().head(10).to_string())


══════════════════════════════════════════════════════════════════════
  1. DATA LOADING & SUMMARY
══════════════════════════════════════════════════════════════════════

── head(5) ──────────────────────────────────────────────────────────
   incident_id                 area_name area_type   latitude  longitude        date time_of_day  hour day_of_week  is_weekend   crime_type  reported_incidents lighting_condition crowd_density police_presence cctv_coverage  proximity_to_police_station_km population_density transport_availability  feel_unsafe_rating weather risk_level  cctv_available  police_patrol  previous_incidents_Monthly  police_station_Distance_km  public_transport_Available  risk_score
0            1        Agha Khan Hospital  hospital  24.892613  67.085916   5/17/2023     morning     7   Wednesday           0      assault                   9           well_lit        medium            none           yes                            3.74             medium                   goo

In [4]:
# ═════════════════════════════════════════════════════════════════════════════
# 2.  FEATURE ENGINEERING & PREPROCESSING
# ═════════════════════════════════════════════════════════════════════════════
section("2. FEATURE ENGINEERING & PREPROCESSING")

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["month"] = df["date"].dt.month
df["year"]  = df["date"].dt.year

# Ordinal encodings
tod_map   = {"morning": 0, "afternoon": 1, "evening": 2, "night": 3}
light_map = {"well_lit": 0, "moderate": 1, "poorly_lit": 2}
crowd_map = {"low": 0, "medium": 1, "high": 2}
pol_map   = {"none": 0, "low": 1, "moderate": 2, "high": 3}
popd_map  = {"low": 0, "medium": 1, "high": 2}
trans_map = {"poor": 0, "moderate": 1, "good": 2}
risk_map  = {"low": 0, "medium": 1, "high": 2}

df["time_of_day_enc"]      = df["time_of_day"].map(tod_map)
df["lighting_enc"]         = df["lighting_condition"].map(light_map)
df["crowd_enc"]            = df["crowd_density"].map(crowd_map)
df["police_enc"]           = df["police_presence"].map(pol_map)
df["pop_density_enc"]      = df["population_density"].map(popd_map)
df["transport_enc"]        = df["transport_availability"].map(trans_map)
df["risk_label"]           = df["risk_level"].map(risk_map)
df["cctv_enc"]             = df["cctv_coverage"].map({"yes": 1, "no": 0})
df["is_night"]             = (df["time_of_day"] == "night").astype(int)
df["is_poorly_lit"]        = (df["lighting_condition"] == "poorly_lit").astype(int)
df["crime_type_enc"]       = LabelEncoder().fit_transform(df["crime_type"])
df["area_type_enc"]        = LabelEncoder().fit_transform(df["area_type"])
df["weather_enc"]          = LabelEncoder().fit_transform(df["weather"])
df["day_of_week_enc"]      = LabelEncoder().fit_transform(df["day_of_week"])

# Risk decay feature — days since incident
df["days_since_incident"]  = (pd.Timestamp("2025-12-31") - df["date"]).dt.days.clip(lower=0)
df["risk_decay_factor"]    = np.exp(-df["days_since_incident"] / 365)

print("  Feature engineering complete.")
print(f"  Total features after engineering: {df.shape[1]}")

FEATURES = [
    "time_of_day_enc", "hour", "lighting_enc", "crowd_enc",
    "police_enc", "pop_density_enc", "transport_enc", "cctv_enc",
    "cctv_available", "police_patrol", "is_weekend", "is_night",
    "is_poorly_lit", "reported_incidents", "feel_unsafe_rating",
    "proximity_to_police_station_km", "previous_incidents_Monthly",
    "police_station_Distance_km", "public_transport_Available",
    "crime_type_enc", "area_type_enc", "weather_enc", "day_of_week_enc",
    "risk_decay_factor", "month",
]

X = df[FEATURES]
y = df["risk_label"]
y_score = df["risk_score"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = np.nan_to_num(X_scaled, nan=0.0)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
_, X_test_score, _, y_test_score = train_test_split(
    X_scaled, y_score, test_size=0.2, random_state=42
)

print(f"\n  Train size: {X_train.shape[0]:,}  |  Test size: {X_test.shape[0]:,}")


══════════════════════════════════════════════════════════════════════
  2. FEATURE ENGINEERING & PREPROCESSING
══════════════════════════════════════════════════════════════════════
  Feature engineering complete.
  Total features after engineering: 46

  Train size: 80,000  |  Test size: 20,000


In [5]:
# ═════════════════════════════════════════════════════════════════════════════
# 3.  EXPLORATORY DATA ANALYSIS — VISUALIZATIONS
# ═════════════════════════════════════════════════════════════════════════════
section("3. EXPLORATORY DATA ANALYSIS — VISUALIZATIONS")

# ── 3A. Overview Dashboard ────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 14), facecolor=PALETTE["bg"])
fig.suptitle("WOMEN SAFETY RISK PREDICTION SYSTEM — KARACHI\nEDA Overview Dashboard",
             fontsize=18, fontweight="bold", color=PALETTE["accent"], y=0.98)
gs = GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.38)

colors_risk = [PALETTE["low"], PALETTE["medium"], PALETTE["high"]]
risk_counts = df["risk_level"].value_counts()[["low", "medium", "high"]]


══════════════════════════════════════════════════════════════════════
  3. EXPLORATORY DATA ANALYSIS — VISUALIZATIONS
══════════════════════════════════════════════════════════════════════


In [6]:
# 1. Risk level pie
ax1 = fig.add_subplot(gs[0, 0])
wedges, texts, autotexts = ax1.pie(
    risk_counts, labels=risk_counts.index,
    autopct="%1.1f%%", colors=colors_risk,
    wedgeprops={"edgecolor": PALETTE["bg"], "linewidth": 2},
    textprops={"color": PALETTE["text"], "fontsize": 10}
)
for at in autotexts: at.set_fontweight("bold")
ax1.set_title("Risk Level Distribution", color=PALETTE["accent"])

Text(0.5, 1.0, 'Risk Level Distribution')

In [7]:
# 2. Crime type bar
ax2 = fig.add_subplot(gs[0, 1])
ct = df["crime_type"].value_counts()
bars = ax2.barh(ct.index, ct.values,
                color=sns.color_palette("magma", len(ct)))
ax2.set_title("Crime Type Frequency", color=PALETTE["accent"])
ax2.set_xlabel("Count")
for bar, v in zip(bars, ct.values):
    ax2.text(v + 100, bar.get_y() + bar.get_height()/2,
             f"{v:,}", va="center", fontsize=8)

In [8]:
# 3. Risk by time of day
ax3 = fig.add_subplot(gs[0, 2])
tod_risk = df.groupby(["time_of_day", "risk_level"]).size().unstack(fill_value=0)
tod_risk = tod_risk.reindex(["morning", "afternoon", "evening", "night"])
tod_risk[["low", "medium", "high"]].plot(
    kind="bar", ax=ax3, color=colors_risk, edgecolor=PALETTE["bg"]
)
ax3.set_title("Risk by Time of Day", color=PALETTE["accent"])
ax3.set_xlabel("Time of Day")
ax3.tick_params(axis="x", rotation=30)
ax3.legend(loc="upper left", fontsize=8)

In [9]:
# 4. Risk by area type
ax4 = fig.add_subplot(gs[0, 3])
at_risk = df.groupby(["area_type", "risk_level"]).size().unstack(fill_value=0)
at_risk[["low", "medium", "high"]].plot(
    kind="bar", ax=ax4, color=colors_risk, edgecolor=PALETTE["bg"]
)
ax4.set_title("Risk by Area Type", color=PALETTE["accent"])
ax4.tick_params(axis="x", rotation=35)
ax4.legend(fontsize=8)

In [10]:
# 5. Risk score distribution
ax5 = fig.add_subplot(gs[1, 0:2])
for lvl, col in zip(["low", "medium", "high"], colors_risk):
    subset = df[df["risk_level"] == lvl]["risk_score"]
    ax5.hist(subset, bins=40, alpha=0.7, color=col, label=lvl, edgecolor=PALETTE["bg"])
ax5.set_title("Risk Score Distribution by Risk Level", color=PALETTE["accent"])
ax5.set_xlabel("Risk Score")
ax5.set_ylabel("Count")
ax5.legend()

In [11]:
# 6. Hourly risk heatmap
ax6 = fig.add_subplot(gs[1, 2:4])
hour_area = df.pivot_table(
    index="area_type", columns="hour", values="risk_score", aggfunc="mean"
)
sns.heatmap(hour_area, ax=ax6, cmap=RISK_CMAP,
            linewidths=0.3, linecolor=PALETTE["bg"],
            cbar_kws={"label": "Avg Risk Score"})
ax6.set_title("Average Risk Score: Area Type × Hour", color=PALETTE["accent"])
ax6.set_xlabel("Hour of Day")

Text(0.5, 525.517094017094, 'Hour of Day')

In [12]:
# 7. Top 10 high-risk areas
ax7 = fig.add_subplot(gs[2, 0:2])
top_areas = (df[df["risk_level"] == "high"]
             .groupby("area_name")["risk_score"].mean()
             .sort_values(ascending=False).head(10))
bars7 = ax7.barh(top_areas.index, top_areas.values,
                 color=PALETTE["high"], edgecolor=PALETTE["bg"])
ax7.set_title("Top 10 High-Risk Areas (Avg Risk Score)", color=PALETTE["accent"])
ax7.set_xlabel("Avg Risk Score")
ax7.invert_yaxis()

In [13]:
# 8. Weekend vs weekday
ax8 = fig.add_subplot(gs[2, 2])
wk = df.groupby(["is_weekend", "risk_level"]).size().unstack(fill_value=0)
wk.index = ["Weekday", "Weekend"]
wk[["low", "medium", "high"]].plot(
    kind="bar", ax=ax8, color=colors_risk, edgecolor=PALETTE["bg"]
)
ax8.set_title("Risk: Weekend vs Weekday", color=PALETTE["accent"])
ax8.tick_params(axis="x", rotation=0)
ax8.legend(fontsize=8)

In [14]:
# 9. Lighting vs risk
ax9 = fig.add_subplot(gs[2, 3])
lt_risk = df.groupby(["lighting_condition", "risk_level"]).size().unstack(fill_value=0)
lt_risk[["low", "medium", "high"]].plot(
    kind="bar", ax=ax9, color=colors_risk, edgecolor=PALETTE["bg"]
)
ax9.set_title("Risk by Lighting Condition", color=PALETTE["accent"])
ax9.tick_params(axis="x", rotation=30)
ax9.legend(fontsize=8)

save(fig, "01_eda_overview_dashboard.png")

  ✔  Saved → 01_eda_overview_dashboard.png


In [15]:
# ── 3B. Correlation Heatmap ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 11), facecolor=PALETTE["bg"])
corr_cols = [
    "risk_score", "risk_label", "time_of_day_enc", "hour", "lighting_enc",
    "crowd_enc", "police_enc", "pop_density_enc", "transport_enc", "cctv_enc",
    "is_night", "is_poorly_lit", "reported_incidents", "feel_unsafe_rating",
    "proximity_to_police_station_km", "previous_incidents_Monthly",
    "risk_decay_factor",
]
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap_corr = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap_corr, center=0,
            square=True, linewidths=0.5, annot=True, fmt=".2f",
            annot_kws={"size": 7}, ax=ax,
            cbar_kws={"shrink": 0.8})
ax.set_title("Feature Correlation Matrix", fontsize=15, color=PALETTE["accent"], pad=15)
fig.suptitle("Women Safety Dataset — Correlation Heatmap", color=PALETTE["text"], y=1.01)
save(fig, "02_correlation_heatmap.png")

  ✔  Saved → 02_correlation_heatmap.png


In [16]:
# ── 3C. Monthly Trend ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor=PALETTE["bg"])
fig.suptitle("Temporal Trends in Crime & Risk", color=PALETTE["accent"], fontsize=14)

monthly = df.groupby(["year", "month"])["risk_score"].mean().reset_index()
monthly["date"] = pd.to_datetime(monthly[["year", "month"]].assign(day=1))
monthly_sorted = monthly.sort_values("date")
axes[0].plot(monthly_sorted["date"], monthly_sorted["risk_score"],
             color=PALETTE["accent"], linewidth=2)
axes[0].fill_between(monthly_sorted["date"], monthly_sorted["risk_score"],
                     alpha=0.2, color=PALETTE["accent"])
axes[0].set_title("Monthly Average Risk Score Over Time")
axes[0].set_xlabel("Date"); axes[0].set_ylabel("Avg Risk Score")

hour_risk = df.groupby("hour")["risk_score"].mean()
axes[1].bar(hour_risk.index, hour_risk.values,
            color=[PALETTE["high"] if h >= 20 or h <= 5
                   else PALETTE["medium"] if h >= 18
                   else PALETTE["low"] for h in hour_risk.index],
            edgecolor=PALETTE["bg"])
axes[1].set_title("Average Risk Score by Hour of Day")
axes[1].set_xlabel("Hour (0–23)"); axes[1].set_ylabel("Avg Risk Score")
axes[1].axvspan(20, 23.5, alpha=0.1, color="red", label="High-risk hours")
axes[1].axvspan(-0.5, 5, alpha=0.1, color="red")
axes[1].legend()
save(fig, "03_temporal_trends.png")

  ✔  Saved → 03_temporal_trends.png


In [17]:
# ── 3D. Geographic Scatter Map ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 9), facecolor=PALETTE["bg"])
for lvl, col, zorder in [("low", PALETTE["low"], 1),
                           ("medium", PALETTE["medium"], 2),
                           ("high", PALETTE["high"], 3)]:
    sub = df[df["risk_level"] == lvl].sample(1500, random_state=42)
    ax.scatter(sub["longitude"], sub["latitude"],
               c=col, alpha=0.4, s=4, label=lvl.upper(), zorder=zorder)
ax.set_title("Karachi Crime Incident Map\n(Colour = Risk Level)",
             color=PALETTE["accent"], fontsize=14)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
legend = ax.legend(markerscale=4, fontsize=10,
                   facecolor=PALETTE["card"], edgecolor=PALETTE["accent"])
for text in legend.get_texts(): text.set_color(PALETTE["text"])
save(fig, "04_geographic_scatter_map.png")


  ✔  Saved → 04_geographic_scatter_map.png


In [18]:
# ── 3E. Feel-Unsafe Rating Analysis ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=PALETTE["bg"])
fig.suptitle("Feel-Unsafe Rating & Crowd Density Analysis", color=PALETTE["accent"])
fur = df.groupby(["feel_unsafe_rating", "risk_level"]).size().unstack(fill_value=0)
fur[["low", "medium", "high"]].plot(kind="bar", ax=axes[0], color=colors_risk,
                                     edgecolor=PALETTE["bg"])
axes[0].set_title("Feel-Unsafe Rating vs Risk Level")
axes[0].set_xlabel("Feel-Unsafe Rating (1–5)")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend()

cd = df.groupby(["crowd_density", "risk_level"]).size().unstack(fill_value=0)
cd.reindex(["low", "medium", "high"])[["low", "medium", "high"]].plot(
    kind="bar", ax=axes[1], color=colors_risk, edgecolor=PALETTE["bg"]
)
axes[1].set_title("Crowd Density vs Risk Level")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend()
save(fig, "05_unsafe_rating_crowd_analysis.png")

  ✔  Saved → 05_unsafe_rating_crowd_analysis.png


In [19]:
# ═════════════════════════════════════════════════════════════════════════════
# 4.  FEATURE SELECTION WITH SelectKBest
# ═════════════════════════════════════════════════════════════════════════════
section("4. FEATURE SELECTION — SelectKBest (ANOVA F-test)")

selector = SelectKBest(score_func=f_classif, k=15)
selector.fit(X_scaled, y)
feat_scores = pd.Series(selector.scores_, index=FEATURES).sort_values(ascending=False)

print("\n  Top 15 features by ANOVA F-score:")
print(feat_scores.head(15).to_string())

fig, ax = plt.subplots(figsize=(12, 6), facecolor=PALETTE["bg"])
top15 = feat_scores.head(15)
bars = ax.barh(top15.index[::-1], top15.values[::-1],
               color=sns.color_palette("plasma", 15)[::-1])
ax.set_title("Feature Importance — SelectKBest (ANOVA F-test)",
             color=PALETTE["accent"], fontsize=14)
ax.set_xlabel("F-Score")
save(fig, "06_feature_importance_kbest.png")

TOP_FEATURES = feat_scores.head(15).index.tolist()
X_top = scaler.fit_transform(df[TOP_FEATURES])
X_top = np.nan_to_num(X_top, nan=0.0)
X_tr, X_te, y_tr, y_te, y_tr_sc, y_te_sc = train_test_split(
    X_top, y, y_score, test_size=0.2, random_state=42, stratify=y
)


══════════════════════════════════════════════════════════════════════
  4. FEATURE SELECTION — SelectKBest (ANOVA F-test)
══════════════════════════════════════════════════════════════════════

  Top 15 features by ANOVA F-score:
time_of_day_enc               23989.433336
is_night                      15011.233237
lighting_enc                   8782.066779
feel_unsafe_rating             6615.715681
reported_incidents             5799.367819
is_poorly_lit                  3298.255781
police_enc                     2461.495361
crowd_enc                      1241.994590
cctv_enc                        720.950504
crime_type_enc                  277.666995
hour                            263.999681
area_type_enc                   215.539553
previous_incidents_Monthly        2.615887
weather_enc                       2.279271
police_patrol                     2.262193
  ✔  Saved → 06_feature_importance_kbest.png


In [20]:
# ═════════════════════════════════════════════════════════════════════════════
# 5.  PCA — DIMENSIONALITY REDUCTION
# ═════════════════════════════════════════════════════════════════════════════
section("5. PCA — DIMENSIONALITY REDUCTION")

pca = PCA(n_components=10, random_state=42)
X_pca_full = pca.fit_transform(X_scaled)

print("\n  Explained Variance Ratio (top 10 PCs):")
for i, ev in enumerate(pca.explained_variance_ratio_, 1):
    bar = "█" * int(ev * 200)
    print(f"  PC{i:>2}: {ev:.4f}  {bar}")
print(f"\n  Cumulative variance (10 PCs): {pca.explained_variance_ratio_.sum():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=PALETTE["bg"])
fig.suptitle("PCA — Principal Component Analysis", color=PALETTE["accent"])


══════════════════════════════════════════════════════════════════════
  5. PCA — DIMENSIONALITY REDUCTION
══════════════════════════════════════════════════════════════════════

  Explained Variance Ratio (top 10 PCs):
  PC 1: 0.0941  ██████████████████
  PC 2: 0.0661  █████████████
  PC 3: 0.0584  ███████████
  PC 4: 0.0514  ██████████
  PC 5: 0.0468  █████████
  PC 6: 0.0430  ████████
  PC 7: 0.0420  ████████
  PC 8: 0.0410  ████████
  PC 9: 0.0408  ████████
  PC10: 0.0407  ████████

  Cumulative variance (10 PCs): 0.5245


Text(0.5, 0.98, 'PCA — Principal Component Analysis')

In [21]:
# Scree plot
evr = pca.explained_variance_ratio_
axes[0].bar(range(1, len(evr)+1), evr, color=PALETTE["accent"])
axes[0].plot(range(1, len(evr)+1), np.cumsum(evr), "o-",
             color="#f1c40f", label="Cumulative")
axes[0].set_title("Scree Plot — Explained Variance")
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Variance Explained")
axes[0].legend()

In [22]:
# PC1 vs PC2 scatter
sample_idx = np.random.choice(len(X_pca_full), 3000, replace=False)
Xp = X_pca_full[sample_idx]
yp = y.values[sample_idx]
c_map = {0: PALETTE["low"], 1: PALETTE["medium"], 2: PALETTE["high"]}
colors_pca = [c_map[v] for v in yp]
axes[1].scatter(Xp[:, 0], Xp[:, 1], c=colors_pca, alpha=0.4, s=5)
axes[1].set_title("PCA: PC1 vs PC2 (colour = Risk Level)")
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
patches = [mpatches.Patch(color=c_map[i], label=l)
           for i, l in enumerate(["Low", "Medium", "High"])]
axes[1].legend(handles=patches)
save(fig, "07_pca_analysis.png")

  ✔  Saved → 07_pca_analysis.png


In [23]:

# ═════════════════════════════════════════════════════════════════════════════
# 6.  MODEL TRAINING & EVALUATION
# ═════════════════════════════════════════════════════════════════════════════
section("6. MODEL TRAINING & EVALUATION")

results = {}
label_names = ["Low", "Medium", "High"]


══════════════════════════════════════════════════════════════════════
  6. MODEL TRAINING & EVALUATION
══════════════════════════════════════════════════════════════════════


In [24]:

# ── 6A. Logistic Regression ───────────────────────────────────────────────────
print("\n  [A] Logistic Regression")
lr = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
lr.fit(X_tr, y_tr)
y_pred_lr = lr.predict(X_te)
acc_lr = accuracy_score(y_te, y_pred_lr)
cv_lr  = cross_val_score(lr, X_top, y, cv=5, scoring="accuracy").mean()
results["Logistic Regression"] = {"acc": acc_lr, "cv": cv_lr, "pred": y_pred_lr}
print(f"    Accuracy : {acc_lr:.4f}")
print(f"    CV Mean  : {cv_lr:.4f}")
print(classification_report(y_te, y_pred_lr, target_names=label_names))


  [A] Logistic Regression
    Accuracy : 0.8372
    CV Mean  : 0.8341
              precision    recall  f1-score   support

         Low       0.80      0.72      0.76      2307
      Medium       0.76      0.79      0.78      7127
        High       0.90      0.90      0.90     10566

    accuracy                           0.84     20000
   macro avg       0.82      0.80      0.81     20000
weighted avg       0.84      0.84      0.84     20000



In [25]:
# ── 6B. K-Nearest Neighbors ───────────────────────────────────────────────────
print("\n  [B] K-Nearest Neighbors (k=7)")
knn = KNeighborsClassifier(n_neighbors=7, metric="euclidean", n_jobs=-1)
knn.fit(X_tr, y_tr)
y_pred_knn = knn.predict(X_te)
acc_knn = accuracy_score(y_te, y_pred_knn)
cv_knn  = cross_val_score(knn, X_top, y, cv=5, scoring="accuracy").mean()
results["KNN (k=7)"] = {"acc": acc_knn, "cv": cv_knn, "pred": y_pred_knn}
print(f"    Accuracy : {acc_knn:.4f}")
print(f"    CV Mean  : {cv_knn:.4f}")
print(classification_report(y_te, y_pred_knn, target_names=label_names))


  [B] K-Nearest Neighbors (k=7)
    Accuracy : 0.7880
    CV Mean  : 0.7895
              precision    recall  f1-score   support

         Low       0.69      0.65      0.67      2307
      Medium       0.69      0.74      0.72      7127
        High       0.88      0.85      0.87     10566

    accuracy                           0.79     20000
   macro avg       0.76      0.75      0.75     20000
weighted avg       0.79      0.79      0.79     20000



In [27]:
# ── 6C. Random Forest ────────────────────────────────────────────────────────
print("\n  [C] Random Forest (100 trees)")
rf = RandomForestClassifier(n_estimators=100, max_depth=12,
                             random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)
y_pred_rf = rf.predict(X_te)
acc_rf = accuracy_score(y_te, y_pred_rf)
cv_rf  = cross_val_score(rf, X_top, y, cv=5, scoring="accuracy").mean()
results["Random Forest"] = {"acc": acc_rf, "cv": cv_rf, "pred": y_pred_rf}
print(f"    Accuracy : {acc_rf:.4f}")
print(f"    CV Mean  : {cv_rf:.4f}")
print(classification_report(y_te, y_pred_rf, target_names=label_names))


  [C] Random Forest (100 trees)
    Accuracy : 0.8515
    CV Mean  : 0.8492
              precision    recall  f1-score   support

         Low       0.86      0.63      0.73      2307
      Medium       0.77      0.84      0.80      7127
        High       0.91      0.91      0.91     10566

    accuracy                           0.85     20000
   macro avg       0.85      0.79      0.81     20000
weighted avg       0.86      0.85      0.85     20000



In [28]:
# ── 6D. Linear Regression (risk score) ───────────────────────────────────────
print("\n  [D] Linear Regression (Continuous Risk Score)")
lin = LinearRegression()
lin.fit(X_tr, y_tr_sc)
y_pred_lin = lin.predict(X_te)
rmse = mean_squared_error(y_te_sc, y_pred_lin) ** 0.5
r2   = r2_score(y_te_sc, y_pred_lin)
print(f"    RMSE : {rmse:.4f}")
print(f"    R²   : {r2:.4f}")


  [D] Linear Regression (Continuous Risk Score)
    RMSE : 0.1134
    R²   : 0.3825


In [29]:
# ═════════════════════════════════════════════════════════════════════════════
# 7.  MODEL COMPARISON VISUALIZATIONS
# ═════════════════════════════════════════════════════════════════════════════
section("7. MODEL COMPARISON VISUALIZATIONS")

# ── 7A. Confusion matrices side by side ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=PALETTE["bg"])
fig.suptitle("Confusion Matrices — All Classification Models",
             color=PALETTE["accent"], fontsize=15)
for ax, (name, res) in zip(axes, {
    "Logistic Regression": results["Logistic Regression"]["pred"],
    "KNN (k=7)":           results["KNN (k=7)"]["pred"],
    "Random Forest":       results["Random Forest"]["pred"],
}.items()):
    cm = confusion_matrix(y_te, res)
    disp = ConfusionMatrixDisplay(cm, display_labels=label_names)
    disp.plot(ax=ax, cmap="YlOrRd", colorbar=False)
    ax.set_title(name, color=PALETTE["accent"])
    ax.set_facecolor("#111122")
save(fig, "08_confusion_matrices.png")


══════════════════════════════════════════════════════════════════════
  7. MODEL COMPARISON VISUALIZATIONS
══════════════════════════════════════════════════════════════════════
  ✔  Saved → 08_confusion_matrices.png


In [30]:
# ── 7B. Model accuracy bar chart ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5), facecolor=PALETTE["bg"])
names = list(results.keys())
accs  = [results[n]["acc"] for n in names]
cvs   = [results[n]["cv"]  for n in names]
x = np.arange(len(names))
w = 0.35
bars1 = ax.bar(x - w/2, accs, w, color=PALETTE["accent"], label="Test Accuracy",
               edgecolor=PALETTE["bg"])
bars2 = ax.bar(x + w/2, cvs,  w, color="#3498db", label="CV Mean Accuracy",
               edgecolor=PALETTE["bg"])
for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylim(0.5, 1.0)
ax.set_title("Model Performance Comparison", color=PALETTE["accent"], fontsize=14)
ax.set_ylabel("Accuracy")
ax.legend()
save(fig, "09_model_comparison.png")

  ✔  Saved → 09_model_comparison.png


In [31]:
# ── 7C. ROC curves (Random Forest — best model) ───────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7), facecolor=PALETTE["bg"])
y_te_bin  = label_binarize(y_te, classes=[0, 1, 2])
y_prob_rf = rf.predict_proba(X_te)
roc_colors = [PALETTE["low"], PALETTE["medium"], PALETTE["high"]]
for i, (cls, col) in enumerate(zip(label_names, roc_colors)):
    fpr, tpr, _ = roc_curve(y_te_bin[:, i], y_prob_rf[:, i])
    auc = roc_auc_score(y_te_bin[:, i], y_prob_rf[:, i])
    ax.plot(fpr, tpr, color=col, lw=2, label=f"{cls} (AUC={auc:.3f})")
ax.plot([0,1],[0,1], "--", color="gray", lw=1)
ax.set_title("ROC Curves — Random Forest", color=PALETTE["accent"], fontsize=14)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.legend(fontsize=11)
save(fig, "10_roc_curves.png")

  ✔  Saved → 10_roc_curves.png


In [32]:
# ── 7D. RF Feature Importance ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6), facecolor=PALETTE["bg"])
fi = pd.Series(rf.feature_importances_, index=TOP_FEATURES).sort_values()
fi.plot(kind="barh", ax=ax, color=sns.color_palette("plasma", len(fi)))
ax.set_title("Random Forest — Feature Importance", color=PALETTE["accent"], fontsize=14)
ax.set_xlabel("Importance Score")
save(fig, "11_rf_feature_importance.png")

  ✔  Saved → 11_rf_feature_importance.png


In [33]:
# ── 7E. Linear Regression — Actual vs Predicted ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=PALETTE["bg"])
fig.suptitle("Linear Regression — Risk Score Prediction", color=PALETTE["accent"])
actual = y_te_sc.values
predicted = y_pred_lin
axes[0].scatter(actual, predicted, alpha=0.2, s=3, color=PALETTE["accent"])
mn, mx = min(actual.min(), predicted.min()), max(actual.max(), predicted.max())
axes[0].plot([mn, mx], [mn, mx], "r--", lw=1.5, label="Perfect fit")
axes[0].set_title(f"Actual vs Predicted  (R²={r2:.3f})")
axes[0].set_xlabel("Actual Risk Score"); axes[0].set_ylabel("Predicted Risk Score")
axes[0].legend()
residuals = actual - predicted
axes[1].hist(residuals, bins=60, color=PALETTE["medium"], edgecolor=PALETTE["bg"])
axes[1].axvline(0, color="red", linestyle="--", lw=1.5)
axes[1].set_title(f"Residual Distribution  (RMSE={rmse:.4f})")
axes[1].set_xlabel("Residual"); axes[1].set_ylabel("Count")
save(fig, "12_linear_regression_results.png")


  ✔  Saved → 12_linear_regression_results.png


In [34]:
# ═════════════════════════════════════════════════════════════════════════════
# 8.  DBSCAN — CRIME HOTSPOT CLUSTERING  [EXTRA FEATURE 1]
# ═════════════════════════════════════════════════════════════════════════════
section("8. DBSCAN — CRIME HOTSPOT CLUSTERING  [EXTRA]")

# Work on HIGH-risk incidents only
high_df = df[df["risk_level"] == "high"].sample(5000, random_state=42)
coords = high_df[["latitude", "longitude"]].values
# eps in degrees ≈ ~300m
db = DBSCAN(eps=0.003, min_samples=15, n_jobs=-1).fit(coords)
high_df = high_df.copy()
high_df["cluster"] = db.labels_

n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
n_noise    = (db.labels_ == -1).sum()
print(f"\n  Clusters found : {n_clusters}")
print(f"  Noise points   : {n_noise}")

cluster_summary = (high_df[high_df["cluster"] >= 0]
                   .groupby("cluster")
                   .agg(count=("risk_score", "count"),
                        avg_risk=("risk_score", "mean"),
                        lat=("latitude", "mean"),
                        lon=("longitude", "mean"))
                   .sort_values("count", ascending=False))
print("\n  Top 10 Hotspot Clusters:")
print(cluster_summary.head(10).to_string())

fig, ax = plt.subplots(figsize=(13, 10), facecolor=PALETTE["bg"])
noise = high_df[high_df["cluster"] == -1]
ax.scatter(noise["longitude"], noise["latitude"],
           c="gray", alpha=0.15, s=3, label="Noise")
cluster_ids = sorted(high_df[high_df["cluster"] >= 0]["cluster"].unique())
palette_cl  = plt.cm.tab20(np.linspace(0, 1, min(len(cluster_ids), 20)))
for cid, col in zip(cluster_ids[:20], palette_cl):
    sub = high_df[high_df["cluster"] == cid]
    ax.scatter(sub["longitude"], sub["latitude"],
               c=[col], alpha=0.6, s=8)
    cx, cy = sub["longitude"].mean(), sub["latitude"].mean()
    ax.annotate(f"C{cid}", (cx, cy), color="white",
                fontsize=7, ha="center",
                bbox=dict(boxstyle="round,pad=0.2", fc="#e94560", alpha=0.7))
ax.set_title("DBSCAN Crime Hotspot Clustering — Karachi\n(High-Risk Incidents Only)",
             color=PALETTE["accent"], fontsize=14)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
save(fig, "13_dbscan_hotspot_map.png")


══════════════════════════════════════════════════════════════════════
  8. DBSCAN — CRIME HOTSPOT CLUSTERING  [EXTRA]
══════════════════════════════════════════════════════════════════════

  Clusters found : 32
  Noise points   : 5

  Top 10 Hotspot Clusters:
         count  avg_risk        lat        lon
cluster                                       
0         1313  0.498857  24.863785  67.014975
9          387  0.504107  24.876186  67.060750
1          279  0.493260  24.812625  67.028516
8          201  0.493640  24.934066  67.111749
15         187  0.520567  24.910312  67.029335
26         130  0.496013  24.950039  67.299912
18         129  0.476240  25.019604  67.110165
22         128  0.519967  24.906464  67.160747
10         116  0.495373  24.900379  66.999942
20         112  0.473888  24.959556  67.078260
  ✔  Saved → 13_dbscan_hotspot_map.png


In [35]:
# ═════════════════════════════════════════════════════════════════════════════
# 9.  SAFE ROUTE RECOMMENDATION  [EXTRA FEATURE 2]
# ═════════════════════════════════════════════════════════════════════════════
section("9. SAFE ROUTE RECOMMENDATION  [EXTRA]")

def area_risk_profile(area_name: str, time_of_day: str) -> dict:
    """Return avg risk metrics for an area at a given time."""
    sub = df[(df["area_name"] == area_name) &
             (df["time_of_day"] == time_of_day)]
    if sub.empty:
        sub = df[df["area_name"] == area_name]
    return {
        "area": area_name,
        "time": time_of_day,
        "avg_risk_score": sub["risk_score"].mean(),
        "high_pct": (sub["risk_level"] == "high").mean() * 100,
        "incidents": sub["reported_incidents"].mean(),
        "dominant_crime": sub["crime_type"].mode()[0] if not sub.empty else "N/A",
    }

def compare_routes(route_a: list, route_b: list, time_of_day: str = "night"):
    """Compare two routes (lists of area names) and recommend the safer one."""
    print(f"\n  ── Route Comparison at [{time_of_day.upper()}] ──────────────────────────")
    scores_a, scores_b = [], []
    print(f"\n  {'ROUTE A':<40} {'AVG RISK':>10}  {'HIGH%':>8}")
    for area in route_a:
        p = area_risk_profile(area, time_of_day)
        scores_a.append(p["avg_risk_score"])
        print(f"    {area:<38} {p['avg_risk_score']:>10.3f}  {p['high_pct']:>7.1f}%")

    print(f"\n  {'ROUTE B':<40} {'AVG RISK':>10}  {'HIGH%':>8}")
    for area in route_b:
        p = area_risk_profile(area, time_of_day)
        scores_b.append(p["avg_risk_score"])
        print(f"    {area:<38} {p['avg_risk_score']:>10.3f}  {p['high_pct']:>7.1f}%")

    avg_a, avg_b = np.mean(scores_a), np.mean(scores_b)
    print(f"\n  Route A overall score: {avg_a:.3f}")
    print(f"  Route B overall score: {avg_b:.3f}")
    winner = "A" if avg_a < avg_b else "B"
    diff   = abs(avg_a - avg_b) / max(avg_a, avg_b) * 100
    print(f"\n  ✅  RECOMMENDED: Route {winner} is {diff:.1f}% safer.")
    return scores_a, scores_b, avg_a, avg_b, winner

route_a = ["DHA Phase 6", "Clifton", "Saddar"]
route_b = ["Lyari Expressway", "Orangi Town", "Baldia Town"]

scores_a, scores_b, avg_a, avg_b, winner = compare_routes(route_a, route_b, "night")


══════════════════════════════════════════════════════════════════════
  9. SAFE ROUTE RECOMMENDATION  [EXTRA]
══════════════════════════════════════════════════════════════════════

  ── Route Comparison at [NIGHT] ──────────────────────────

  ROUTE A                                    AVG RISK     HIGH%
    DHA Phase 6                                 0.494     84.6%
    Clifton                                     0.486     83.0%
    Saddar                                      0.492     89.5%

  ROUTE B                                    AVG RISK     HIGH%
    Lyari Expressway                            0.489     94.8%
    Orangi Town                                 0.499     81.7%
    Baldia Town                                 0.499     82.4%

  Route A overall score: 0.491
  Route B overall score: 0.496

  ✅  RECOMMENDED: Route A is 1.0% safer.


In [36]:
# Route comparison chart
fig, ax = plt.subplots(figsize=(11, 5), facecolor=PALETTE["bg"])
x_a = np.arange(len(route_a))
x_b = np.arange(len(route_b))
ax.plot(x_a, scores_a, "o-", color="#3498db", lw=2.5, ms=8, label="Route A (DHA→Saddar)")
ax.plot(x_b, scores_b, "s-", color=PALETTE["high"], lw=2.5, ms=8, label="Route B (Lyari→Baldia)")
ax.axhline(avg_a, color="#3498db", linestyle="--", alpha=0.5, lw=1)
ax.axhline(avg_b, color=PALETTE["high"], linestyle="--", alpha=0.5, lw=1)
ax.set_title("Safe Route Comparison — Night Travel Risk",
             color=PALETTE["accent"], fontsize=14)
ax.set_ylabel("Avg Risk Score")
ax.set_xticks(x_a); ax.set_xticklabels([f"Stop {i+1}" for i in x_a])
ax.legend()
ax.text(0.02, 0.95, f"✅ Route {'A' if winner=='A' else 'B'} RECOMMENDED",
        transform=ax.transAxes, color="#2ecc71", fontsize=12, fontweight="bold",
        va="top")
save(fig, "14_safe_route_comparison.png")

  ✔  Saved → 14_safe_route_comparison.png


In [37]:
# ═════════════════════════════════════════════════════════════════════════════
# 10. TIME-BASED RISK ALERT  [EXTRA FEATURE 3]
# ═════════════════════════════════════════════════════════════════════════════
section("10. TIME-BASED RISK ALERT  [EXTRA]")

def time_risk_alert(area_name: str):
    """Show how risk multiplies across time periods for an area."""
    tod_risk = (df[df["area_name"] == area_name]
                .groupby("time_of_day")["risk_score"]
                .mean()
                .reindex(["morning", "afternoon", "evening", "night"]))
    base = tod_risk["morning"]
    print(f"\n  Area: {area_name}")
    print(f"  {'Time':<12}  {'Avg Risk':>10}  {'Multiplier':>12}  Alert")
    for t, v in tod_risk.items():
        mult = v / base
        alert = ""
        if mult >= 2.5:    alert = "🚨 CRITICAL — Avoid if possible"
        elif mult >= 1.5:  alert = "⚠️  HIGH — Extra caution"
        elif mult >= 1.1:  alert = "🟡 MODERATE"
        else:              alert = "🟢 SAFER"
        print(f"  {t:<12}  {v:>10.3f}  {mult:>11.2f}x  {alert}")
    return tod_risk

print("\n  Time-based risk alerts for key areas:")
for area in ["Lyari Expressway", "Clifton", "Tower Bus Stop", "DHA Phase 6"]:
    time_risk_alert(area)


══════════════════════════════════════════════════════════════════════
  10. TIME-BASED RISK ALERT  [EXTRA]
══════════════════════════════════════════════════════════════════════

  Time-based risk alerts for key areas:

  Area: Lyari Expressway
  Time            Avg Risk    Multiplier  Alert
  morning            0.492         1.00x  🟢 SAFER
  afternoon          0.494         1.00x  🟢 SAFER
  evening            0.490         0.99x  🟢 SAFER
  night              0.489         0.99x  🟢 SAFER

  Area: Clifton
  Time            Avg Risk    Multiplier  Alert
  morning            0.497         1.00x  🟢 SAFER
  afternoon          0.493         0.99x  🟢 SAFER
  evening            0.486         0.98x  🟢 SAFER
  night              0.486         0.98x  🟢 SAFER

  Area: Tower Bus Stop
  Time            Avg Risk    Multiplier  Alert
  morning            0.487         1.00x  🟢 SAFER
  afternoon          0.493         1.01x  🟢 SAFER
  evening            0.483         0.99x  🟢 SAFER
  night           

In [38]:
# Alert visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor=PALETTE["bg"])
fig.suptitle("Time-Based Risk Alerts — Area Comparison",
             color=PALETTE["accent"], fontsize=15)
alert_areas = ["Lyari Expressway", "Clifton", "Tower Bus Stop", "Saddar"]
tod_order   = ["morning", "afternoon", "evening", "night"]
bar_colors  = [PALETTE["low"], "#f1c40f", PALETTE["medium"], PALETTE["high"]]
for ax, area in zip(axes.flatten(), alert_areas):
    tod_r = (df[df["area_name"] == area]
             .groupby("time_of_day")["risk_score"]
             .mean().reindex(tod_order))
    bars = ax.bar(tod_order, tod_r.values, color=bar_colors, edgecolor=PALETTE["bg"])
    for bar, v in zip(bars, tod_r.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f"{v:.3f}", ha="center", fontsize=9)
    ax.set_title(area, color=PALETTE["accent"])
    ax.set_ylabel("Avg Risk Score")
    ax.set_ylim(0, 0.85)
save(fig, "15_time_based_risk_alerts.png")

  ✔  Saved → 15_time_based_risk_alerts.png


In [39]:
# ═════════════════════════════════════════════════════════════════════════════
# 11. RISK DECAY SIMULATION  [EXTRA FEATURE 4]
# ═════════════════════════════════════════════════════════════════════════════
section("11. RISK DECAY SIMULATION  [EXTRA]")

def risk_decay(base_score: float, days_since: int,
               half_life: int = 90) -> float:
    """Exponential decay: risk halves every `half_life` days with no incidents."""
    lam = np.log(2) / half_life
    return base_score * np.exp(-lam * days_since)

days   = np.arange(0, 365)
scores = [0.85, 0.65, 0.45]
labels = ["High-Risk Start", "Medium-Risk Start", "Low-Risk Start"]
colors_d = [PALETTE["high"], PALETTE["medium"], PALETTE["low"]]

fig, ax = plt.subplots(figsize=(12, 6), facecolor=PALETTE["bg"])
for base, lbl, col in zip(scores, labels, colors_d):
    decayed = [risk_decay(base, d) for d in days]
    ax.plot(days, decayed, color=col, lw=2.5, label=lbl)
ax.axhline(0.35, color="yellow", linestyle="--", lw=1, alpha=0.7,
           label="Medium→Low threshold (0.35)")
ax.axhline(0.60, color="orange", linestyle="--", lw=1, alpha=0.7,
           label="High→Medium threshold (0.60)")
ax.fill_between(days, 0, 0.35, alpha=0.08, color=PALETTE["low"])
ax.fill_between(days, 0.35, 0.60, alpha=0.08, color=PALETTE["medium"])
ax.fill_between(days, 0.60, 1.0, alpha=0.08, color=PALETTE["high"])
ax.set_title("Risk Decay Over Time (No New Incidents)\nHalf-life = 90 days",
             color=PALETTE["accent"], fontsize=14)
ax.set_xlabel("Days Since Last Incident")
ax.set_ylabel("Risk Score")
ax.legend(fontsize=9)
save(fig, "16_risk_decay_simulation.png")

print("\n  Risk Decay Examples:")
print(f"  Area with score 0.85 after 30d:  {risk_decay(0.85,30):.3f}")
print(f"  Area with score 0.85 after 90d:  {risk_decay(0.85,90):.3f}")
print(f"  Area with score 0.85 after 180d: {risk_decay(0.85,180):.3f}")
print(f"  Area with score 0.85 after 365d: {risk_decay(0.85,365):.3f}")


══════════════════════════════════════════════════════════════════════
  11. RISK DECAY SIMULATION  [EXTRA]
══════════════════════════════════════════════════════════════════════
  ✔  Saved → 16_risk_decay_simulation.png

  Risk Decay Examples:
  Area with score 0.85 after 30d:  0.675
  Area with score 0.85 after 90d:  0.425
  Area with score 0.85 after 180d: 0.212
  Area with score 0.85 after 365d: 0.051


In [40]:
# ═════════════════════════════════════════════════════════════════════════════
# 12. RISK PREDICTION FUNCTION (live inference)
# ═════════════════════════════════════════════════════════════════════════════
section("12. LIVE RISK PREDICTION FUNCTION")

RISK_LABELS = {0: "🟢 LOW", 1: "🟡 MEDIUM", 2: "🔴 HIGH"}

def predict_risk(area_name, area_type, time_of_day, hour,
                 lighting, crowd, police, cctv, weather="clear",
                 feel_unsafe=2, pop_density="medium",
                 transport="moderate", is_weekend=0):
    """Predict risk level for given inputs using Random Forest."""
    tod_map2   = {"morning":0,"afternoon":1,"evening":2,"night":3}
    light_map2 = {"well_lit":0,"moderate":1,"poorly_lit":2}
    crowd_map2 = {"low":0,"medium":1,"high":2}
    pol_map2   = {"none":0,"low":1,"moderate":2,"high":3}
    popd_map2  = {"low":0,"medium":1,"high":2}
    trans_map2 = {"poor":0,"moderate":1,"good":2}

    # Pull area stats from dataset
    area_sub = df[df["area_name"] == area_name] if area_name in df["area_name"].values else df
    prev_inc = area_sub["previous_incidents_Monthly"].mean()
    pol_dist = area_sub["police_station_Distance_km"].mean()
    rep_inc  = area_sub["reported_incidents"].mean()
    decay    = 0.8  # assume recent activity

    at_enc  = LabelEncoder().fit(df["area_type"]).transform([area_type])[0]
    wx_enc  = LabelEncoder().fit(df["weather"]).transform([weather])[0]
    dow_enc = 2  # Wednesday default

    row = pd.DataFrame([{
        "time_of_day_enc":               tod_map2.get(time_of_day, 3),
        "hour":                          hour,
        "lighting_enc":                  light_map2.get(lighting, 2),
        "crowd_enc":                     crowd_map2.get(crowd, 1),
        "police_enc":                    pol_map2.get(police, 0),
        "pop_density_enc":               popd_map2.get(pop_density, 1),
        "transport_enc":                 trans_map2.get(transport, 1),
        "cctv_enc":                      1 if cctv == "yes" else 0,
        "cctv_available":                1 if cctv == "yes" else 0,
        "police_patrol":                 1 if police != "none" else 0,
        "is_weekend":                    is_weekend,
        "is_night":                      1 if time_of_day == "night" else 0,
        "is_poorly_lit":                 1 if lighting == "poorly_lit" else 0,
        "reported_incidents":            rep_inc,
        "feel_unsafe_rating":            feel_unsafe,
        "proximity_to_police_station_km": pol_dist,
        "previous_incidents_Monthly":    prev_inc,
        "police_station_Distance_km":    pol_dist,
        "public_transport_Available":    1 if transport != "poor" else 0,
        "crime_type_enc":                2,
        "area_type_enc":                 at_enc,
        "weather_enc":                   wx_enc,
        "day_of_week_enc":               dow_enc,
        "risk_decay_factor":             decay,
        "month":                         6,
    }])

    row_scaled = scaler.transform(row[TOP_FEATURES])
    pred_label = rf.predict(row_scaled)[0]
    pred_proba = rf.predict_proba(row_scaled)[0]

    print(f"\n  ── Prediction for: {area_name} | {time_of_day.upper()} ──")
    print(f"     Risk Level : {RISK_LABELS[pred_label]}")
    print(f"     Confidence : Low={pred_proba[0]:.2f}  "
          f"Medium={pred_proba[1]:.2f}  High={pred_proba[2]:.2f}")
    advice = {
        0: "Area appears relatively safe. Standard precautions advised.",
        1: "Moderate risk. Stay alert, share location, avoid isolated spots.",
        2: "HIGH RISK. Avoid travel alone. Use trusted transport. Call helpline if needed.",
    }
    print(f"     Advice     : {advice[pred_label]}")
    return RISK_LABELS[pred_label], pred_proba



══════════════════════════════════════════════════════════════════════
  12. LIVE RISK PREDICTION FUNCTION
══════════════════════════════════════════════════════════════════════


In [41]:
# Demo predictions
print("\n  Demo Predictions:")
predict_risk("Lyari Expressway",  "road",        "night",   23, "poorly_lit", "low",  "none",     "no",  feel_unsafe=5)
predict_risk("DHA Phase 6",       "residential", "morning",  8, "well_lit",   "low",  "high",     "yes", feel_unsafe=1)
predict_risk("Tower Bus Stop",    "bus_stop",    "evening", 19, "moderate",   "high", "low",      "no",  feel_unsafe=3)
predict_risk("Agha Khan Hospital","hospital",    "night",    1, "well_lit",   "medium","moderate","yes", feel_unsafe=2)


  Demo Predictions:

  ── Prediction for: Lyari Expressway | NIGHT ──
     Risk Level : 🔴 HIGH
     Confidence : Low=0.00  Medium=0.00  High=1.00
     Advice     : HIGH RISK. Avoid travel alone. Use trusted transport. Call helpline if needed.

  ── Prediction for: DHA Phase 6 | MORNING ──
     Risk Level : 🟢 LOW
     Confidence : Low=0.95  Medium=0.05  High=0.00
     Advice     : Area appears relatively safe. Standard precautions advised.

  ── Prediction for: Tower Bus Stop | EVENING ──
     Risk Level : 🔴 HIGH
     Confidence : Low=0.00  Medium=0.02  High=0.98
     Advice     : HIGH RISK. Avoid travel alone. Use trusted transport. Call helpline if needed.

  ── Prediction for: Agha Khan Hospital | NIGHT ──
     Risk Level : 🟡 MEDIUM
     Confidence : Low=0.09  Medium=0.79  High=0.12
     Advice     : Moderate risk. Stay alert, share location, avoid isolated spots.


('🟡 MEDIUM', array([0.08682046, 0.78912216, 0.12405738]))